In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("API KEY SET")

API KEY SET


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

llm.invoke("Hello!").content

f:\AI_Projects\rag_basics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'Hello. How can I assist you today?'

## **RAG IMPLEMENTATION WITH PDF's** 

## STEP 1 - Extracting Text From PDF's

In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/Module 1.pdf"

loader = PyPDFLoader(pdf_path)

docs = loader.load()

docs

[Document(metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2020-09-03T13:51:05+08:00', 'subject': '', 'author': 'CBIT-ECE', 'keywords': '', 'moddate': '2020-11-25T00:01:10+05:30', 'company': '', 'comments': '', 'sourcemodified': "D:20200903135105+08'21'", 'title': 'MANAGEMENT & ENTREPRENEURSHIP', 'trapped': '/False', 'source': './Docs/Module 1.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1'}, page_content='VTUPulse.com \n \n \n \nMANAGEMENT & ENTREPRENEURSHIP MODULE 1\nCBIT-KOLAR Page 1\nTechnological Innovation\nManagement & Entrepreneurship\nB.E., V Semester, Electronics & Communication\nEngineering\n[As per Choice Based Credit System (CBCS) scheme]\n[Subject Code: 18ES51]\nMODULE – 1a\nMANAGEMENT\nDefinition of management:\n \n \n \n \n \nVTUPulse.com'),
 Document(metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2020-09-03T13:51:05+08:00', 'subject': '', 'author': 'CBIT-ECE', 'keywords': '', 'moddate': '2020-11-25T00:01:10+05:30', 'company

## STEP 2 - Splitting the Document into chunks

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)

chunks = splitter.split_documents(docs)

chunks

[Document(metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2020-09-03T13:51:05+08:00', 'subject': '', 'author': 'CBIT-ECE', 'keywords': '', 'moddate': '2020-11-25T00:01:10+05:30', 'company': '', 'comments': '', 'sourcemodified': "D:20200903135105+08'21'", 'title': 'MANAGEMENT & ENTREPRENEURSHIP', 'trapped': '/False', 'source': './Docs/Module 1.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1'}, page_content='VTUPulse.com \n \n \n \nMANAGEMENT & ENTREPRENEURSHIP MODULE 1\nCBIT-KOLAR Page 1\nTechnological Innovation\nManagement & Entrepreneurship\nB.E., V Semester, Electronics & Communication\nEngineering\n[As per Choice Based Credit System (CBCS) scheme]\n[Subject Code: 18ES51]\nMODULE – 1a\nMANAGEMENT\nDefinition of management:\n \n \n \n \n \nVTUPulse.com'),
 Document(metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2020-09-03T13:51:05+08:00', 'subject': '', 'author': 'CBIT-ECE', 'keywords': '', 'moddate': '2020-11-25T00:01:10+05:30', 'company

In [7]:
len(chunks)

73

In [14]:
#PyPDF Loader Will Also Create Metadata

chunks[0].metadata

{'producer': '',
 'creator': 'WPS Writer',
 'creationdate': '2020-09-03T13:51:05+08:00',
 'subject': '',
 'author': 'CBIT-ECE',
 'keywords': '',
 'moddate': '2020-11-25T00:01:10+05:30',
 'company': '',
 'comments': '',
 'sourcemodified': "D:20200903135105+08'21'",
 'title': 'MANAGEMENT & ENTREPRENEURSHIP',
 'trapped': '/False',
 'source': './Docs/Module 1.pdf',
 'total_pages': 32,
 'page': 0,
 'page_label': '1'}

## **Creating Own MetaData For PDF Chunks**

In [12]:
for i in docs:

    i.metadata = {"source" : "Module 1.pdf",
                  "developer" : "Microsoft"}

In [13]:
docs

[Document(metadata={'source': 'Module 1.pdf', 'developer': 'Microsoft'}, page_content='VTUPulse.com \n \n \n \nMANAGEMENT & ENTREPRENEURSHIP MODULE 1\nCBIT-KOLAR Page 1\nTechnological Innovation\nManagement & Entrepreneurship\nB.E., V Semester, Electronics & Communication\nEngineering\n[As per Choice Based Credit System (CBCS) scheme]\n[Subject Code: 18ES51]\nMODULE – 1a\nMANAGEMENT\nDefinition of management:\n \n \n \n \n \nVTUPulse.com'),
 Document(metadata={'source': 'Module 1.pdf', 'developer': 'Microsoft'}, page_content='VTUPulse.com \n \n \n \nMANAGEMENT & ENTREPRENEURSHIP MODULE 1\nCBIT-KOLAR Page 2\nManagement is a function of guidance and leadership control of efforts of a\ngroup or individuals in order to achieve goals/objectives of an organization.\nSimplest definition is that it is defined as the art of getting things done\nthrough people. Management can also be defined as The process consisting of\nplanning, organizing, actuating, and controlling performed to determine and

## STEP 3 - Creating Embeddings for the chunks

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12845.69it/s]


## STEP 4 - Create & Store Embeddings in Vector Store

In [9]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

## STEP 5 - Semantic Search

In [16]:
vectorstore.similarity_search("What are the functions of management?", k=5)

[Document(metadata={'page': 6, 'author': 'CBIT-ECE', 'source': './Docs/Module 1.pdf', 'total_pages': 32, 'comments': '', 'producer': '', 'sourcemodified': "D:20200903135105+08'21'", 'subject': '', 'creator': 'WPS Writer', 'trapped': '/False', 'moddate': '2020-11-25T00:01:10+05:30', 'company': '', 'title': 'MANAGEMENT & ENTREPRENEURSHIP', 'page_label': '7', 'creationdate': '2020-09-03T13:51:05+08:00', 'keywords': ''}, page_content='well managed.\nSo, management is required in all aspects of life.\nFUNCTIONS OF MANAGEMENT\nThough many authors have defined several functions of management,\nthere are five essential and well accepted functions of management. They are:\n> Planning.\n> Organising.\n> Staffing.\n> Directing (leading) and\n> Controlling.\nPLANNING:\nPlanning is an executive function that is referred to as decision making.\nIt involves missions and objectives and the actions to achieve them. This\n \n \n \n \n \nVTUPulse.com'),
 Document(metadata={'subject': '', 'creationdate': 

## TALK TO LLM

In [17]:
context = vectorstore.similarity_search("What are the functions of management?", k=5)

response = llm.invoke(f"What are the functions of management? You can answer using the following context: {context}")

print(response.content)

According to the provided documents, the five essential and well-accepted functions of management are:

1. **Planning**: This is an executive function that involves decision-making, missions, objectives, and the actions to achieve them. Planning is referred to as decision-making and involves setting goals and objectives and the actions to achieve them.

2. **Organising**: This function involves putting together the necessary resources, including money, men, material, and machines, to achieve the goals and objectives of the organization.

3. **Staffing**: This function involves selecting, training, and developing the employees to achieve the goals and objectives of the organization.

4. **Directing (Leading)**: This function involves guiding and motivating the employees to achieve the goals and objectives of the organization.

5. **Controlling**: This function involves monitoring and evaluating the progress towards the goals and objectives of the organization and taking corrective actio